# Fleet encoder pretraining — Colab

Trains the single-fleet encoder (`agents/transformer_v1/encoder/fleet_encoder.py`) on the multi-task `ENCODER_PRETRAIN_LABELS` objective using CSVs under `data/datasets/fleet/`.

**Before running**: upload the repo (or its `data/datasets/fleet/` + `agents/` subset) to Google Drive, e.g. at `MyDrive/orbit-wars/`.

**Outputs** land in `data/runs/fleet/<timestamp>/`: `fleet_encoder_best.pt`, `fleet_encoder_last.pt`, `log.json`, `test_summary.json`.

GPU runtime is recommended (Runtime → Change runtime type → GPU).

## 1. Mount Drive and locate the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO = '/content/drive/MyDrive/orbit-wars'  # adjust if you uploaded elsewhere
assert os.path.isdir(REPO), f'repo not found at {REPO}'
os.chdir(REPO)
sys.path.insert(0, REPO)
print('cwd =', os.getcwd())

## 2. Sanity check the dataset

In [ ]:
import json
from pathlib import Path
data_dir = Path('data/datasets/fleet')
manifest = json.loads((data_dir / 'manifest.json').read_text())
for split, files in manifest.items():
    if isinstance(files, list):
        print(f'{split}: {len(files)} CSVs')
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 3. Train

`train(...)` runs the full multi-task loop, saves `.pt` checkpoints each epoch, writes `log.json`, and prints per-head val metrics every `analyze_every` epochs. Tweak `epochs`, `d_model`, `batch_size` as needed.

In [ ]:
import time
from pathlib import Path
from agents.transformer_v1.encoder.pretrain import train

out_dir = Path('data/runs/fleet') / time.strftime('%Y%m%d-%H%M%S')
best_ckpt = train(
    data_dir=Path('data/datasets/fleet'),
    out_dir=out_dir,
    d_model=64,
    batch_size=2048,
    epochs=50,
    lr=1e-3,
    weight_decay=1e-4,
    eval_every=1,
    analyze_every=5,
    num_workers=2,
)
print('best ckpt:', best_ckpt)

## 4. Plot training curves

Per-head val loss/acc over epochs. Useful for catching heads that stalled while the mean loss looks fine.

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

log = json.loads((out_dir / 'log.json').read_text())
epochs = [e['epoch'] for e in log if 'val' in e]
head_names = list(log[-1]['val'].keys())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name in head_names:
    losses = [e['val'][name]['loss'] for e in log if 'val' in e]
    axes[0].plot(epochs, losses, label=name, alpha=0.8)
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('val loss'); axes[0].set_yscale('log')
axes[0].legend(fontsize=7, ncol=2); axes[0].set_title('per-head val loss')

for name in head_names:
    if 'acc' not in log[-1]['val'][name]:
        continue
    accs = [e['val'][name]['acc'] for e in log if 'val' in e]
    axes[1].plot(epochs, accs, label=name, alpha=0.8)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val accuracy')
axes[1].legend(fontsize=7); axes[1].set_title('per-head val accuracy')
plt.tight_layout(); plt.show()

## 5. Test-set summary

In [ ]:
summary = json.loads((out_dir / 'test_summary.json').read_text())
for name, m in summary.items():
    extra = f"  acc={m['acc']:.3f}" if 'acc' in m else ''
    print(f'{name:<28s}  loss={m["loss"]:.4f}{extra}')